In [7]:
import tkinter as tk
from tkinter import messagebox, scrolledtext
from datetime import datetime, timedelta
import urllib.request
import json
import statistics
import random
import webbrowser
import threading
import time


class GoldQuantDashboard:
    def __init__(self, root):
        self.root = root
        self.root.title("Gold Quant Interactive Dashboard")
        self.root.geometry("1100x750")

        self.latency_history = []

        tk.Label(
            root,
            text="Gold Quant Analytics Engine",
            font=("Arial", 20, "bold"),
            fg="#C71585"
        ).pack(pady=10)

        tk.Label(
            root,
            text=(
                "Simulates gold prices, evaluates sentiment sensitivity, "
                "forecasts short-term markouts, runs Monte Carlo projections, "
                "calculates risk metrics, monitors pipeline latency, "
                "and visualizes results in an interactive dashboard."
            ),
            wraplength=1000,
            font=("Arial", 10)
        ).pack(pady=5)

        frame = tk.Frame(root)
        frame.pack(pady=10)

        tk.Label(frame, text="Ticker:").grid(row=0, column=0)
        self.ticker_entry = tk.Entry(frame, width=12)
        self.ticker_entry.insert(0, "GC=F")
        self.ticker_entry.grid(row=0, column=1, padx=5)

        tk.Label(frame, text="Start Date:").grid(row=0, column=2)
        self.start_entry = tk.Entry(frame, width=15)
        self.start_entry.insert(0, "2024-01-01")
        self.start_entry.grid(row=0, column=3, padx=5)

        tk.Label(frame, text="End Date:").grid(row=0, column=4)
        self.end_entry = tk.Entry(frame, width=15)
        self.end_entry.insert(0, "2025-01-01")
        self.end_entry.grid(row=0, column=5, padx=5)

        self.run_button = tk.Button(
            frame,
            text="Run Quant Dashboard",
            command=self.start_thread,
            bg="#DB7093",
            fg="white",
            font=("Arial", 11, "bold")
        )
        self.run_button.grid(row=0, column=6, padx=10)

        tk.Button(
            frame,
            text="Open Yahoo Finance",
            command=self.open_yahoo,
            bg="#8B008B",
            fg="white",
            font=("Arial", 11, "bold")
        ).grid(row=0, column=7, padx=5)

        self.canvas = tk.Canvas(
            root,
            width=1050,
            height=300,
            bg="white"
        )
        self.canvas.pack(pady=10)

        self.output = scrolledtext.ScrolledText(
            root,
            width=130,
            height=20,
            font=("Consolas", 10)
        )
        self.output.pack(padx=10, pady=10)

    def open_yahoo(self):
        ticker = self.ticker_entry.get().strip()
        webbrowser.open(f"https://finance.yahoo.com/quote/{ticker}")

    def safe_output(self, text):
        self.output.delete("1.0", tk.END)
        self.output.insert(tk.END, text)

    def start_thread(self):
        self.run_button.config(state="disabled")
        self.safe_output("Running full quant dashboard...\n")

        thread = threading.Thread(target=self.run_model)
        thread.daemon = True
        thread.start()

    def date_to_unix(self, text):
        return int(datetime.strptime(text, "%Y-%m-%d").timestamp())

    def download_yahoo_data(self, ticker, start_date, end_date):
        period1 = self.date_to_unix(start_date)
        period2 = self.date_to_unix(end_date)

        url = (
            f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}"
            f"?period1={period1}&period2={period2}&interval=1d"
        )

        request = urllib.request.Request(
            url,
            headers={"User-Agent": "Mozilla/5.0"}
        )

        with urllib.request.urlopen(request, timeout=15) as response:
            raw = response.read().decode("utf-8")

        data = json.loads(raw)
        result = data["chart"]["result"][0]

        timestamps = result["timestamp"]
        closes = result["indicators"]["quote"][0]["close"]

        rows = []

        for ts, close in zip(timestamps, closes):
            if close is not None:
                rows.append({
                    "date": datetime.fromtimestamp(ts).strftime("%Y-%m-%d"),
                    "close": float(close),
                    "sentiment": random.uniform(-1, 1)
                })

        return rows

    def calculate_returns(self, prices):
        returns = []

        for i in range(1, len(prices)):
            if prices[i - 1] != 0:
                returns.append((prices[i] - prices[i - 1]) / prices[i - 1])

        return returns

    def sentiment_sensitivity(self, prices, sentiments):
        mean_price = statistics.mean(prices)
        mean_sentiment = statistics.mean(sentiments)

        numerator = sum(
            (s - mean_sentiment) * (p - mean_price)
            for s, p in zip(sentiments, prices)
        )

        denominator = sum(
            (s - mean_sentiment) ** 2
            for s in sentiments
        )

        if denominator == 0:
            return 0

        return numerator / denominator

    def markout_forecast(self, prices, days=5):
        recent = prices[-15:]
        avg_move = statistics.mean(
            recent[i] - recent[i - 1]
            for i in range(1, len(recent))
        )

        return prices[-1] + avg_move * days

    def monte_carlo(self, spot, avg_return):
        simulations = []

        for _ in range(300):
            future = spot

            for _ in range(30):
                shock = random.uniform(-0.015, 0.015)
                future *= 1 + avg_return + shock

            simulations.append(future)

        return simulations

    def calculate_risk(self, returns):
        sorted_returns = sorted(returns)
        var_95 = sorted_returns[int(len(sorted_returns) * 0.05)]

        tail = [r for r in returns if r <= var_95]
        expected_shortfall = statistics.mean(tail) if tail else 0

        volatility = statistics.stdev(returns) if len(returns) > 1 else 0

        sharpe = (
            statistics.mean(returns) / volatility * (252 ** 0.5)
            if volatility != 0 else 0
        )

        return var_95, expected_shortfall, volatility, sharpe

    def draw_dashboard(self, rows, simulations, markout_price):
        self.canvas.delete("all")

        prices = [row["close"] for row in rows[-80:]]
        dates = [row["date"] for row in rows[-80:]]

        min_price = min(prices)
        max_price = max(prices)

        width = 1000
        height = 220
        x_start = 30
        y_start = 250

        points = []

        for i, price in enumerate(prices):
            x = x_start + i * (width / max(1, len(prices) - 1))
            y = y_start - ((price - min_price) / (max_price - min_price + 0.0001)) * height
            points.append((x, y))

        for i in range(1, len(points)):
            self.canvas.create_line(
                points[i - 1][0],
                points[i - 1][1],
                points[i][0],
                points[i][1],
                width=2
            )

        self.canvas.create_text(
            530,
            20,
            text="Interactive Dashboard: Gold Price, Markout Forecast, Monte Carlo Range",
            font=("Arial", 14, "bold")
        )

        self.canvas.create_text(
            80,
            280,
            text=dates[0],
            font=("Arial", 9)
        )

        self.canvas.create_text(
            980,
            280,
            text=dates[-1],
            font=("Arial", 9)
        )

        latest_x, latest_y = points[-1]
        self.canvas.create_oval(
            latest_x - 5,
            latest_y - 5,
            latest_x + 5,
            latest_y + 5,
            fill="#C71585"
        )

        self.canvas.create_text(
            latest_x - 60,
            latest_y - 20,
            text=f"Spot ${prices[-1]:,.2f}",
            font=("Arial", 9, "bold")
        )

        mc_low = min(simulations)
        mc_high = max(simulations)

        self.canvas.create_text(
            250,
            50,
            text=f"5-Day Markout: ${markout_price:,.2f}",
            font=("Arial", 11)
        )

        self.canvas.create_text(
            550,
            50,
            text=f"Monte Carlo Low: ${mc_low:,.2f}",
            font=("Arial", 11)
        )

        self.canvas.create_text(
            830,
            50,
            text=f"Monte Carlo High: ${mc_high:,.2f}",
            font=("Arial", 11)
        )

    def run_model(self):
        start_time = time.perf_counter()

        try:
            ticker = self.ticker_entry.get().strip()
            start_date = self.start_entry.get().strip()
            end_date = self.end_entry.get().strip()

            data_arrival_time = time.perf_counter()

            rows = self.download_yahoo_data(ticker, start_date, end_date)

            if len(rows) < 30:
                raise ValueError("Please choose at least 30 days of data.")

            prices = [row["close"] for row in rows]
            sentiments = [row["sentiment"] for row in rows]
            returns = self.calculate_returns(prices)

            spot = prices[-1]
            avg_return = statistics.mean(returns)

            beta = self.sentiment_sensitivity(prices, sentiments)
            markout = self.markout_forecast(prices)
            simulations = self.monte_carlo(spot, avg_return)

            expected = statistics.mean(simulations)
            low = min(simulations)
            high = max(simulations)

            var_95, es, volatility, sharpe = self.calculate_risk(returns)

            end_time = time.perf_counter()

            network_latency = (data_arrival_time - start_time) * 1000
            software_latency = (end_time - data_arrival_time) * 1000
            total_latency = network_latency + software_latency

            self.latency_history.append(total_latency)

            report = f"""
======================================================================
GOLD QUANT ANALYTICS ENGINE REPORT
======================================================================

PROJECT DESCRIPTION
----------------------------------------------------------------------
This application simulates gold prices, evaluates sentiment sensitivity,
forecasts short-term markouts, runs Monte Carlo projections, calculates
risk metrics, monitors pipeline latency, and visualizes results in an
interactive dashboard.

DATA SOURCE
----------------------------------------------------------------------
Ticker: {ticker}
Yahoo Finance:
https://finance.yahoo.com/quote/{ticker}

Date Range: {start_date} to {end_date}
Rows Loaded: {len(rows)}

MARKET SUMMARY
----------------------------------------------------------------------
Latest Gold Price: ${spot:,.2f}
5-Day Markout Forecast: ${markout:,.2f}

SENTIMENT SENSITIVITY
----------------------------------------------------------------------
Synthetic Sentiment Beta: {beta:,.4f}

MONTE CARLO PROJECTION
----------------------------------------------------------------------
Expected 30-Day Price: ${expected:,.2f}
Low Case: ${low:,.2f}
High Case: ${high:,.2f}

RISK METRICS
----------------------------------------------------------------------
Average Daily Return: {avg_return:.4%}
Daily Volatility: {volatility:.4%}
VaR 95%: {var_95:.4%}
Expected Shortfall: {es:.4%}
Sharpe Ratio: {sharpe:.2f}

PIPELINE LATENCY
----------------------------------------------------------------------
Network / Data Latency: {network_latency:.2f} ms
Software Processing Latency: {software_latency:.2f} ms
Total Pipeline Latency: {total_latency:.2f} ms
Average Latency: {statistics.mean(self.latency_history):.2f} ms

LAST 10 PRICES
----------------------------------------------------------------------
"""

            for row in rows[-10:]:
                report += f"{row['date']} | Close: ${row['close']:,.2f} | Sentiment: {row['sentiment']:.3f}\n"

            self.root.after(0, lambda: self.safe_output(report))
            self.root.after(0, lambda: self.draw_dashboard(rows, simulations, markout))
            self.root.after(0, lambda: self.run_button.config(state="normal"))

        except Exception as e:
            self.root.after(0, lambda: messagebox.showerror("Error", str(e)))
            self.root.after(0, lambda: self.run_button.config(state="normal"))


if __name__ == "__main__":
    root = tk.Tk()
    app = GoldQuantDashboard(root)
    root.mainloop()